# BTR-Transformer

**73.69 Large Language Models (2026) — Trabajo Práctico 1**

Impression-level binary classification of `bought` as a product-level contribution to supermarket e-commerce **Buy Through Rate** (purchases / impressions). No decision threshold. Ranking quality of bought=1 is measured with ROC-AUC and PR-AUC.

Architecture: a **small Transformer encoder trained from scratch** on product text (`title`, `description`, `ingredients`), concatenated with a tabular vector, then an MLP → 1 logit. BERT is used **only as a tokenizer**; encoder weights are random `nn.Embedding`, not a pretrained LLM backbone.

This notebook covers setup through the training machinery (cell groups 0–7). Baselines, the three ablation arms, scale-up, and Exercise 3 are placeholders at the end.

## Locked constraints

- **`cart` is leakage**, not a feature. \(P(bought \mid cart=	ext{false}) = 0\); using it would change the task from BTR among impressions to conversion among cart-adds. See [`adr/0002`](../adr/0002-leakage-cart-query-id.md).
- **`query_id` is a split grouping key**, never a model input. Impressions from the same search must not leak across train/valid/test. See [`adr/0005`](../adr/0005-grouped-splits.md).
- The catalog text **encodes the target** via a behavioural cue in `title` (and restated in `description`). Arm A vs C measures cue visibility; **B vs C** is the only pair that speaks to self-attention on product language. See [`adr/0009`](../adr/0009-text-behavioural-cue.md).

Design: [`docs/DESIGN.md`](../docs/DESIGN.md) · data facts: [`docs/EDA.md`](../docs/EDA.md) · decisions: [`docs/OPEN_DECISIONS.md`](../docs/OPEN_DECISIONS.md) · ADRs: [`adr/`](../adr/).


## 1. Environment and data loading

Setup is a **shell** step, not a notebook cell (D21):

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

GPU is used if present; this dataset trains on CPU in minutes ([`DESIGN.md`](../docs/DESIGN.md) §8). The next cell asserts the pinned versions from `requirements.txt` and fails loudly on mismatch rather than installing.


In [19]:
from __future__ import annotations

from importlib.metadata import version as _pkg_version

EXPECTED = {
    "torch": "2.13.0",
    "pandas": "2.3.3",
    "scikit-learn": "1.7.2",
    "transformers": "5.16.1",
}


def _base_version(v: str) -> str:
    return v.split("+")[0]


for dist, want in EXPECTED.items():
    got = _base_version(_pkg_version(dist))
    assert got == want, f"{dist}: expected {want}, got {got}"

print("version assertions OK:", {k: _base_version(_pkg_version(k)) for k in EXPECTED})


version assertions OK: {'torch': '2.13.0', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'transformers': '5.16.1'}


In [20]:
import inspect
import math
import os
import random
import re
from collections.abc import Sequence
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import constants as hf_constants
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

# --- Hyperparameters ---
SEED = 42
BATCH_SIZE = 64
MAX_LENGTH = 80
MAX_EPOCHS = 30
PATIENCE = 5
LR = 3e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
ARM_SEEDS: tuple[int, ...] = (42, 43, 44, 45, 46)


# --- Reproducibility ---
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(SEED)

# --- Device ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"device={device}")
print(f"HF_HOME={hf_constants.HF_HOME}")
print(f"HF_HUB_CACHE={hf_constants.HF_HUB_CACHE}")

# --- Tokenizer (WordPiece vocab only; encoder weights stay random) ---
try:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", local_files_only=True)
except OSError:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"tokenizer={tokenizer.name_or_path} vocab_size={tokenizer.vocab_size} sep={tokenizer.sep_token!r}")


device=cpu
HF_HOME=/home/lcampoli/.cache/huggingface
HF_HUB_CACHE=/home/lcampoli/.cache/huggingface/hub
tokenizer=bert-base-uncased vocab_size=30522 sep='[SEP]'


### CSV schema 

Default path: `data/raw/supermarket_products.csv`. Override with env `BTR_DATA_PATH` (absolute or repo-relative).

Required columns (names + roles). Row count, missingness rates, and class balance are **data-dependent** — printed, not asserted to fixed values.

| Role | Columns |
| --- | --- |
| Target | `bought` |
| Leakage drop | `cart` |
| Split key | `query_id` |
| Text | `title`, `description`, `ingredients` |
| Numeric / parse | `price`, `net_weight_oz`, `nutrition_score`, `dimensions_in`, `package_size` |
| Categorical | `category`, `brand`, `storage_type`, `unit_of_measure`, `country_of_origin`, `allergens` |
| Filters | `filter_category`, `filter_storage_type`, `filter_price_min`, `filter_price_max` |
| Time | `timestamp` |

Structural checks on load: required columns present; `bought` / `cart` boolean-like; every purchase implies `cart=True` (else `cart` is not pure leakage); ≥2 queries and both purchase regimes for stratified grouped split.


In [21]:
REQUIRED_COLUMNS: frozenset[str] = frozenset(
    {
        "bought",
        "cart",
        "query_id",
        "title",
        "description",
        "ingredients",
        "price",
        "net_weight_oz",
        "nutrition_score",
        "dimensions_in",
        "package_size",
        "category",
        "brand",
        "storage_type",
        "unit_of_measure",
        "country_of_origin",
        "allergens",
        "filter_category",
        "filter_storage_type",
        "filter_price_min",
        "filter_price_max",
        "timestamp",
    }
)
DEFAULT_CSV_REL = Path("data") / "raw" / "supermarket_products.csv"


def resolve_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "requirements.txt").exists() and (cand / "data" / "raw").is_dir():
            return cand
        if (cand / DEFAULT_CSV_REL).exists():
            return cand
    raise FileNotFoundError(
        "Could not find repo root (requirements.txt + data/raw/) from cwd or parents."
    )


def resolve_data_path() -> Path:
    override = os.environ.get("BTR_DATA_PATH")
    if override:
        p = Path(override).expanduser()
        return p if p.is_absolute() else (Path.cwd() / p).resolve()
    return resolve_repo_root() / DEFAULT_CSV_REL


def validate_raw_frame(frame: pd.DataFrame) -> None:
    missing = REQUIRED_COLUMNS - set(frame.columns)
    assert not missing, f"missing required columns: {sorted(missing)}"
    extra = set(frame.columns) - REQUIRED_COLUMNS
    if extra:
        print(f"note: extra columns ignored later: {sorted(extra)}")

    assert len(frame) >= 2, "need at least 2 rows"
    bought = frame["bought"].astype(int)
    cart = frame["cart"].astype(int)
    assert set(bought.unique()) <= {0, 1}, "bought must be binary / boolean-like"
    assert set(cart.unique()) <= {0, 1}, "cart must be binary / boolean-like"
    n_leak = int((~frame["cart"].astype(bool) & frame["bought"].astype(bool)).sum())
    assert n_leak == 0, (
        f"{n_leak} rows have bought=True with cart=False; cart is not pure leakage"
    )

    n_queries = int(frame["query_id"].nunique())
    assert n_queries >= 2, "need ≥2 distinct query_id for a grouped split"
    has_purchase = frame.groupby("query_id")["bought"].max().astype(int)
    assert has_purchase.nunique() == 2, (
        "stratified query split needs both purchase and no-purchase queries"
    )


REPO_ROOT = resolve_repo_root()
DATA_PATH = resolve_data_path()
print(f"REPO_ROOT={REPO_ROOT}")
print(f"DATA_PATH={DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)
validate_raw_frame(df_raw)

print("shape", df_raw.shape)
print("dtypes:\n", df_raw.dtypes.to_string())
print("\nmissingness (%):\n", df_raw.isna().mean().mul(100).round(2).to_string())
print(f"\nbought rate: {df_raw['bought'].astype(float).mean():.4f}")
print("cart × bought:")
print(pd.crosstab(df_raw["cart"], df_raw["bought"]))
print(f"n query_id: {df_raw['query_id'].nunique()}")
print(df_raw.groupby("query_id").size().describe().to_string())
print("schema OK (invariants only; no row-count / rate fingerprints)")


REPO_ROOT=/home/lcampoli/proyectos/tp1-llm
DATA_PATH=/home/lcampoli/proyectos/tp1-llm/data/raw/supermarket_products.csv
shape (10000, 22)
dtypes:
 title                   object
description             object
price                  float64
category                object
timestamp               object
query_id                object
filter_category         object
filter_price_min       float64
filter_price_max       float64
filter_storage_type     object
cart                      bool
bought                    bool
brand                   object
package_size            object
unit_of_measure         object
net_weight_oz          float64
dimensions_in           object
storage_type            object
ingredients             object
allergens               object
nutrition_score          int64
country_of_origin       object

missingness (%):
 title                   0.00
description             0.00
price                   0.00
category                0.00
timestamp               0.00
query_i

In [22]:
def split_queries(
    frame: pd.DataFrame,
    seed: int,
    train_frac: float = 0.70,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Grouped 70/15/15 split of query_id, stratified on has-≥1-purchase."""
    labels = frame.groupby("query_id")["bought"].max().astype(int)
    ids = labels.index.to_numpy()
    y = labels.to_numpy()
    train_ids, rest_ids, _, rest_y = train_test_split(
        ids, y, train_size=train_frac, random_state=seed, stratify=y
    )
    valid_ids, test_ids = train_test_split(
        rest_ids, test_size=0.5, random_state=seed, stratify=rest_y
    )
    return train_ids, valid_ids, test_ids


TITLE_CUE_RE = re.compile(r"\(([^()]*)\)\s*$")
cue = df_raw["title"].str.extract(TITLE_CUE_RE)[0].fillna("None")
print(f"titles with no parenthetical: {(cue == 'None').sum()}")
print(f"n distinct cue levels (incl. None): {cue.nunique()}")
print("\ncue × bought (purchase rate, n):")
print(
    pd.crosstab(cue, df_raw["bought"], normalize="index")
    .join(cue.value_counts().rename("n"))
    .sort_values("n", ascending=False)
    .to_string()
)

probe_train_ids, _, probe_test_ids = split_queries(df_raw, seed=SEED)
train_mask = df_raw["query_id"].isin(probe_train_ids)
test_mask = df_raw["query_id"].isin(probe_test_ids)
y_test = df_raw.loc[test_mask, "bought"].astype(int).to_numpy()

# Cue-rate probe: P(bought | cue) from train queries → score test.
train_rate = (
    pd.DataFrame({"cue": cue[train_mask], "bought": df_raw.loc[train_mask, "bought"]})
    .groupby("cue")["bought"]
    .mean()
)
cue_pred = cue[test_mask].map(train_rate).fillna(df_raw.loc[train_mask, "bought"].mean())
print(
    f"\ncue-rate probe  ROC-AUC={roc_auc_score(y_test, cue_pred):.3f}  "
    f"PR-AUC={average_precision_score(y_test, cue_pred):.3f}  "
    f"prevalence={y_test.mean():.3f}"
)

# TF-IDF probe: bag-of-words on catalog text → logistic regression.
text_concat = (
    df_raw["title"].astype(str)
    + " "
    + df_raw["description"].astype(str)
    + " "
    + df_raw["ingredients"].astype(str)
)
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3)
clf = LogisticRegression(max_iter=1_000, solver="liblinear")
clf.fit(tfidf.fit_transform(text_concat[train_mask]), df_raw.loc[train_mask, "bought"].astype(int))
tfidf_pred = clf.predict_proba(tfidf.transform(text_concat[test_mask]))[:, 1]
print(
    f"TF-IDF probe    ROC-AUC={roc_auc_score(y_test, tfidf_pred):.3f}  "
    f"PR-AUC={average_precision_score(y_test, tfidf_pred):.3f}"
)


titles with no parenthetical: 511
n distinct cue levels (incl. None): 20

cue × bought (purchase rate, n):
                       False      True    n
0                                          
Discontinuing Soon  1.000000  0.000000  550
Limited Feedback    1.000000  0.000000  524
New Listing         1.000000  0.000000  522
Highly Rated        0.978846  0.021154  520
Unrated Listing     1.000000  0.000000  517
Current Stock       1.000000  0.000000  515
None                1.000000  0.000000  511
Clearance Listing   1.000000  0.000000  510
Shopper Favorite    0.972387  0.027613  507
Standard Listing    1.000000  0.000000  506
Rarely Reordered    1.000000  0.000000  500
#1 Pick             0.375000  0.625000  496
Regular Listing     1.000000  0.000000  494
Customer Favorite   0.322515  0.677485  493
Recently Added      1.000000  0.000000  481
Well Reviewed       0.962264  0.037736  477
Top Rated           0.372881  0.627119  472
Best Seller         0.342553  0.657447  470
Popular Choic

### EDA conclusions

Interpret the **printed** tables above; numbers in [`docs/EDA.md`](../docs/EDA.md) are for the default course CSV only.

- **`cart` leakage:** validated on load — no `bought=True` when `cart=False`. Drop `cart`.
- **Imbalance:** use the printed `bought` rate. Primary metric is PR-AUC; ROC-AUC alongside. No threshold.
- **Filter flags:** compute match flags, log variance, **drop zero-variance columns** (D1). Keep `relative_price_position` only from the filter block.
- **Behavioural cue:** title parentheticals (and mirrored description sentences) can encode purchase propensity. Cue-rate / TF-IDF probes measure text signal before any Transformer claim. Arm A's headline is cue visibility; **B vs C** is the architecture comparison.


## 2. Feature engineering

Dropped before modeling ([`DESIGN.md`](../docs/DESIGN.md) §2):

| Column | Reason |
| --- | --- |
| `cart` | Target leakage |
| `query_id` | Identifier — kept until after the split, then dropped from feature frames |
| `package_size` | Restates `net_weight_oz` + `unit_of_measure` |
| raw `filter_*` | After interaction features are built |
| `dimensions_in` | After `volume` |
| `timestamp` | After cyclical encodings |

Match flags are computed; **zero-variance** ones are dropped (D1). Price-band signal is `relative_price_position` only; zero-width band → `0.5` (guard always defined; non-zero count is logged as a data-quality note).


In [23]:
df = df_raw.copy()

DIM_RE = re.compile(r"([\d.]+)\s*x\s*([\d.]+)\s*x\s*([\d.]+)")
parsed = df["dimensions_in"].str.extract(DIM_RE).astype(float)
assert parsed.notna().all().all(), "dimensions_in parse failed"
df["volume"] = parsed[0] * parsed[1] * parsed[2]
df = df.drop(columns=["dimensions_in"])
print("volume describe:\n", df["volume"].describe().to_string())


volume describe:
 count    10000.000000
mean       243.170959
std        328.786475
min          7.502000
25%         60.192000
50%        109.445000
75%        276.655500
max       3403.536000


In [24]:
df["is_category_match"] = (df["category"] == df["filter_category"]).astype(np.int8)
df["is_storage_match"] = (df["storage_type"] == df["filter_storage_type"]).astype(np.int8)

band_width = df["filter_price_max"] - df["filter_price_min"]
n_zero_width = int((band_width == 0).sum())
print(f"zero-width price bands: {n_zero_width}")
if n_zero_width:
    print("note: zero-width bands mapped to relative_price_position=0.5")
df["relative_price_position"] = np.where(
    band_width == 0,
    0.5,
    (df["price"] - df["filter_price_min"]) / band_width,
)

df = df.drop(
    columns=[
        "filter_category",
        "filter_storage_type",
        "filter_price_min",
        "filter_price_max",
    ]
)


zero-width price bands: 0


In [25]:
match_cols = ["is_category_match", "is_storage_match"]
print("match-flag variance:\n", df[match_cols].var().to_string())
print("nunique", {c: int(df[c].nunique()) for c in match_cols})
# D1: drop zero-variance engineered flags (constant on some CSVs; keep if informative).
drop_match = [c for c in match_cols if df[c].nunique() <= 1]
keep_match = [c for c in match_cols if c not in drop_match]
if drop_match:
    df = df.drop(columns=drop_match)
    print(f"dropped zero-variance match flags: {drop_match}")
if keep_match:
    print(f"kept informative match flags: {keep_match}")


match-flag variance:
 is_category_match    0.0
is_storage_match     0.0
nunique {'is_category_match': 1, 'is_storage_match': 1}
dropped zero-variance match flags: ['is_category_match', 'is_storage_match']


In [26]:
ts = pd.to_datetime(df["timestamp"], utc=True)
hour = ts.dt.hour
day_of_week = ts.dt.dayofweek  # Monday = 0
df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
df["day_sin"] = np.sin(2 * np.pi * day_of_week / 7)
df["day_cos"] = np.cos(2 * np.pi * day_of_week / 7)
df = df.drop(columns=["timestamp"])


In [27]:
df["allergens"] = df["allergens"].fillna("None")
df["y"] = df["bought"].astype(float)
df = df.drop(columns=["cart", "package_size"])
print("allergens levels:", sorted(df["allergens"].unique().tolist()))
print("y mean", df["y"].mean())


allergens levels: ['Fish', 'Milk', 'None', 'Peanuts', 'Shellfish', 'Soy', 'Tree nuts', 'Wheat']
y mean 0.1301


In [28]:
BEHAVIOURAL_SENTENCES = frozenset(
    {
        "A dependable pick according to reviews",
        "Limited customer feedback so far",
        "Stocked as part of the regular lineup",
        "Ships as part of regular inventory",
        "One of the less-repurchased items in its aisle",
        "Feedback on this listing is still limited",
        "Available for standard online ordering",
        "Received mixed feedback from shoppers",
        "Recently added to the online catalog",
        "Rarely reordered by past customers",
        "Generally receives positive feedback",
        "Marked for limited future availability",
        "One of the most repurchased items in its aisle",
        "Frequently reordered by returning customers",
        "A newer addition to the catalog",
        "Often recommended by repeat customers",
        "Consistently praised in customer feedback",
        "Rated highly by shoppers for consistent quality",
        "Well liked by regular shoppers",
    }
)


def strip_title_cue(title: str) -> str:
    return TITLE_CUE_RE.sub("", title).rstrip()


def _sentences(text: str) -> list[str]:
    return [p.strip().rstrip(".") for p in re.split(r"\.\s*", text.strip()) if p.strip()]


def strip_description_cue(description: str) -> tuple[str, list[str]]:
    parts = _sentences(description)
    removed = [p for p in parts if p in BEHAVIOURAL_SENTENCES]
    kept = [p for p in parts if p not in BEHAVIOURAL_SENTENCES]
    rebuilt = ". ".join(kept)
    if rebuilt:
        rebuilt += "."
    return rebuilt, removed


stripped_titles: list[str] = []
stripped_descs: list[str] = []
all_removed: list[str] = []
for title, desc in zip(df["title"], df["description"], strict=True):
    stripped_titles.append(strip_title_cue(title))
    new_desc, removed = strip_description_cue(desc)
    stripped_descs.append(new_desc)
    all_removed.extend(removed)

assert set(all_removed) <= BEHAVIOURAL_SENTENCES
listed_before = [
    s for d in df["description"] for s in _sentences(d) if s.startswith("Listed under ")
]
listed_after = [
    s for d in stripped_descs for s in _sentences(d) if s.startswith("Listed under ")
]
assert listed_before == listed_after
print(f"removed behavioural description sentences: {len(all_removed)}")
print(f"surviving 'Listed under …' sentences: {len(listed_after)}")
if not listed_after:
    print("note: no surviving 'Listed under …' sentences in this CSV")

ws_before = int((df["title"] + " " + df["description"]).str.split().str.len().sum())
ws_after = int(
    pd.Series(stripped_titles).str.split().str.len().sum()
    + pd.Series(stripped_descs).str.split().str.len().sum()
)
print(f"whitespace tokens title+description: before={ws_before} after={ws_after} delta={ws_before - ws_after}")

sep = tokenizer.sep_token
df["text_full"] = (
    df["title"].astype(str)
    + f" {sep} "
    + df["description"].astype(str)
    + f" {sep} "
    + df["ingredients"].astype(str)
)
df["text_stripped"] = (
    pd.Series(stripped_titles, index=df.index)
    + f" {sep} "
    + pd.Series(stripped_descs, index=df.index)
    + f" {sep} "
    + df["ingredients"].astype(str)
)

print("\n--- before / after sample ---")
i = 0
print("FULL:", df["text_full"].iloc[i][:400])
print("STRIPPED:", df["text_stripped"].iloc[i][:400])
assert TITLE_CUE_RE.search(stripped_titles[i]) is None or stripped_titles[i] == df["title"].iloc[i]


removed behavioural description sentences: 9541
surviving 'Listed under …' sentences: 10000
whitespace tokens title+description: before=367580 after=292929 delta=74651

--- before / after sample ---
FULL: Cedar House Steamable Pepperoni Pizza - 10 oz (Well Reviewed) [SEP] Steamable pepperoni pizza in a 10 oz package for online grocery orders. Listed under frozen and intended for frozen storage. Generally receives positive feedback. [SEP] Prepared ingredients, Spices, Salt
STRIPPED: Cedar House Steamable Pepperoni Pizza - 10 oz [SEP] Steamable pepperoni pizza in a 10 oz package for online grocery orders. Listed under frozen and intended for frozen storage. [SEP] Prepared ingredients, Spices, Salt


In [31]:
print("engineered columns:", list(df.columns))
print(df.head(2).T)
print("\nvolume skew (raw):", float(df["volume"].skew()))
print("volume skew (log1p):", float(np.log1p(df["volume"]).skew()))
print("query_id still present:", "query_id" in df.columns)


engineered columns: ['title', 'description', 'price', 'category', 'query_id', 'bought', 'brand', 'unit_of_measure', 'net_weight_oz', 'storage_type', 'ingredients', 'allergens', 'nutrition_score', 'country_of_origin', 'volume', 'relative_price_position', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'y', 'text_full', 'text_stripped']
                                                                         0  \
title                    Cedar House Steamable Pepperoni Pizza - 10 oz ...   
description              Steamable pepperoni pizza in a 10 oz package f...   
price                                                                  8.3   
category                                                            Frozen   
query_id                                                          q_000001   
bought                                                               False   
brand                                                          Cedar House   
unit_of_measure                          

## 3. Train / valid / test split

The unit of splitting is `query_id`, not rows. A row-wise split would put sibling impressions of the same search on both sides of the cut (filters are constant within a query). Queries are **stratified on has-≥1-purchase** (D15) so held-out prevalence stays comparable — PR-AUC's baseline *is* the prevalence. Ratios 70 / 15 / 15, seed 42. No k-fold; dispersion later comes from seed repeats of training (D16), not of the partition.


In [32]:
train_ids, valid_ids, test_ids = split_queries(df, seed=SEED)
id_sets = {"train": set(train_ids), "valid": set(valid_ids), "test": set(test_ids)}
n_queries = int(df["query_id"].nunique())
assert id_sets["train"].isdisjoint(id_sets["valid"])
assert id_sets["train"].isdisjoint(id_sets["test"])
assert id_sets["valid"].isdisjoint(id_sets["test"])
assert len(id_sets["train"] | id_sets["valid"] | id_sets["test"]) == n_queries

splits: dict[str, pd.DataFrame] = {}
qids: dict[str, np.ndarray] = {}
for name, ids in [("train", train_ids), ("valid", valid_ids), ("test", test_ids)]:
    part = df[df["query_id"].isin(ids)].copy()
    qids[name] = part["query_id"].to_numpy()
    splits[name] = part
    print(
        f"{name:5s}  queries={part['query_id'].nunique():4d}  rows={len(part):5d}  "
        f"pos={part['y'].mean():.4f}"
    )

assert sum(len(s) for s in splits.values()) == len(df)
print(f"partition covers all {n_queries} queries and {len(df)} rows")


train  queries=1408  rows= 7032  pos=0.1301
valid  queries= 302  rows= 1471  pos=0.1305
test   queries= 302  rows= 1497  pos=0.1296
partition covers all 2012 queries and 10000 rows


In [33]:
train_df = splits["train"].drop(columns=["query_id"]).reset_index(drop=True)
valid_df = splits["valid"].drop(columns=["query_id"]).reset_index(drop=True)
test_df = splits["test"].drop(columns=["query_id"]).reset_index(drop=True)
qid_train, qid_valid, qid_test = qids["train"], qids["valid"], qids["test"]
assert "query_id" not in train_df.columns
print("query_id dropped from feature frames; aligned arrays retained", 
      qid_train.shape, qid_valid.shape, qid_test.shape)


query_id dropped from feature frames; aligned arrays retained (7032,) (1471,) (1497,)


## 4. Preprocessing pipeline

Every fitted transform (imputer, scaler, one-hot) is fit on **train only** and applied to valid/test. Cue stripping is a fixed rule, already applied identically to all rows. `n_tabular` is read from the fitted encoder at runtime, never hardcoded.


In [34]:
NUMERIC_COLS = ["price", "net_weight_oz", "nutrition_score", "log_volume"]
BOUNDED_COLS = [
    "relative_price_position",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]
# Informative match flags (if not dropped in FE) join the bounded / passthrough block.
for _c in ("is_category_match", "is_storage_match"):
    if _c in train_df.columns:
        BOUNDED_COLS.append(_c)

CATEGORICAL_COLS = [
    "category",
    "storage_type",
    "unit_of_measure",
    "country_of_origin",
    "allergens",
    "brand",
]
print("numeric", NUMERIC_COLS)
print("bounded (no scaler)", BOUNDED_COLS)
print("categorical", CATEGORICAL_COLS)


numeric ['price', 'net_weight_oz', 'nutrition_score', 'log_volume']
bounded (no scaler) ['relative_price_position', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']
categorical ['category', 'storage_type', 'unit_of_measure', 'country_of_origin', 'allergens', 'brand']


In [35]:
for frame in (train_df, valid_df, test_df):
    frame["log_volume"] = np.log1p(frame["volume"])

print(
    f"volume skew train: raw={float(train_df['volume'].skew()):.2f}  "
    f"log1p={float(train_df['log_volume'].skew()):.2f}"
)

tabular_encoder = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            NUMERIC_COLS,
        ),
        ("bound", "passthrough", BOUNDED_COLS),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_COLS,
        ),
    ],
    remainder="drop",
)
tabular_encoder.fit(train_df)

X_train = np.asarray(tabular_encoder.transform(train_df), dtype=np.float32)
X_valid = np.asarray(tabular_encoder.transform(valid_df), dtype=np.float32)
X_test = np.asarray(tabular_encoder.transform(test_df), dtype=np.float32)
y_train = train_df["y"].to_numpy(dtype=np.float32)
y_valid = valid_df["y"].to_numpy(dtype=np.float32)
y_test = test_df["y"].to_numpy(dtype=np.float32)

n_tabular = int(X_train.shape[1])
print(f"n_tabular={n_tabular} (from fitted encoder)")
print("feature names (head):", list(tabular_encoder.get_feature_names_out())[:12], "...")
assert n_tabular > 0
assert X_valid.shape[1] == X_test.shape[1] == n_tabular


volume skew train: raw=2.94  log1p=0.43
n_tabular=62 (from fitted encoder)
feature names (head): ['num__price', 'num__net_weight_oz', 'num__nutrition_score', 'num__log_volume', 'bound__relative_price_position', 'bound__hour_sin', 'bound__hour_cos', 'bound__day_sin', 'bound__day_cos', 'cat__category_Baby', 'cat__category_Bakery', 'cat__category_Beverages'] ...


In [36]:
def encode_text(series: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    enc = tokenizer(
        series.tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=True,
        return_attention_mask=True,
    )
    return (
        np.asarray(enc["input_ids"], dtype=np.int64),
        np.asarray(enc["attention_mask"], dtype=np.int64),
    )


raw_enc = tokenizer(
    df["text_full"].tolist(),
    add_special_tokens=True,
    truncation=False,
    padding=False,
)
wp_lens = np.array([len(ids) for ids in raw_enc["input_ids"]])
print(
    f"WordPiece (full data, incl. specials): "
    f"mean={wp_lens.mean():.1f}  p95={np.percentile(wp_lens, 95):.0f}  max={wp_lens.max()}"
)
n_trunc = int((wp_lens > MAX_LENGTH).sum())
print(f"rows that would truncate at max_length={MAX_LENGTH}: {n_trunc}")
if n_trunc:
    print(
        f"warning: {n_trunc} rows exceed MAX_LENGTH={MAX_LENGTH}; "
        "encode_text truncates them. Consider raising MAX_LENGTH."
    )
else:
    print(f"MAX_LENGTH={MAX_LENGTH} covers all rows (max={int(wp_lens.max())})")


WordPiece (full data, incl. specials): mean=55.6  p95=64  max=74
rows that would truncate at max_length=80: 0
MAX_LENGTH=80 covers all rows (max=74)


In [17]:
print(f"n_tabular={n_tabular}")
print(f"tokenizer vocab_size={tokenizer.vocab_size} pad_id={tokenizer.pad_token_id}")
sample_ids, sample_mask = encode_text(train_df["text_full"].iloc[:1])
print("decoded sample (specials kept):")
print(tokenizer.decode(sample_ids[0].tolist(), skip_special_tokens=False))
print("attention_mask sum", int(sample_mask[0].sum()))


n_tabular=62
tokenizer vocab_size=30522 pad_id=0
decoded sample (specials kept):
[CLS] cedar house steamable pepperoni pizza - 10 oz ( well reviewed ) [SEP] steamable pepperoni pizza in a 10 oz package for online grocery orders. listed under frozen and intended for frozen storage. generally receives positive feedback. [SEP] prepared ingredients, spices, salt [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
attention_mask sum 53


## 5. Dataset and DataLoader

Each item: `input_ids` / `attention_mask` as `LongTensor [80]`, `tabular` as `FloatTensor [n_tabular]`, `label` as scalar `FloatTensor`. Dataset and DataLoader stay on **CPU**; batches move to `device` in the train/eval loop (D12). Default collate is enough — sequences are already padded by the tokenizer. `make_loader(split, text_variant)` is the single code path for Arms A (`full`) and B (`stripped`).


In [18]:
class ImpressionDataset(Dataset):
    """CPU-resident impression tensors. Device transfer happens per batch."""

    def __init__(
        self,
        input_ids: np.ndarray,
        attention_mask: np.ndarray,
        tabular: np.ndarray,
        labels: np.ndarray,
    ) -> None:
        self.input_ids = torch.as_tensor(input_ids, dtype=torch.long)
        self.attention_mask = torch.as_tensor(attention_mask, dtype=torch.long)
        self.tabular = torch.as_tensor(tabular, dtype=torch.float32)
        self.labels = torch.as_tensor(labels, dtype=torch.float32)

    def __len__(self) -> int:
        return int(self.labels.shape[0])

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "tabular": self.tabular[idx],
            "label": self.labels[idx],
        }


In [19]:
_FRAMES = {"train": train_df, "valid": valid_df, "test": test_df}
_TABULAR = {"train": X_train, "valid": X_valid, "test": X_test}
_LABELS = {"train": y_train, "valid": y_valid, "test": y_test}
_TEXT_COL = {"full": "text_full", "stripped": "text_stripped"}
_ENCODE_CACHE: dict[tuple[str, str], tuple[np.ndarray, np.ndarray]] = {}


def make_loader(
    split: str,
    text_variant: str,
    batch_size: int = BATCH_SIZE,
    shuffle: bool | None = None,
) -> DataLoader:
    if shuffle is None:
        shuffle = split == "train"
    key = (split, text_variant)
    if key not in _ENCODE_CACHE:
        col = _TEXT_COL[text_variant]
        _ENCODE_CACHE[key] = encode_text(_FRAMES[split][col])
    ids, mask = _ENCODE_CACHE[key]
    ds = ImpressionDataset(ids, mask, _TABULAR[split], _LABELS[split])
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=torch.cuda.is_available(),
    )


train_loader_a = make_loader("train", "full")
batch_cpu = next(iter(train_loader_a))
print("CPU batch:")
for k, v in batch_cpu.items():
    print(f"  {k:16s} shape={tuple(v.shape)} dtype={v.dtype} device={v.device}")
assert batch_cpu["input_ids"].shape[1] == MAX_LENGTH
assert batch_cpu["tabular"].shape[1] == n_tabular
assert batch_cpu["input_ids"].device.type == "cpu"

batch_dev = {k: v.to(device) for k, v in batch_cpu.items()}
print("after .to(device):", {k: str(v.device) for k, v in batch_dev.items()})


CPU batch:
  input_ids        shape=(64, 80) dtype=torch.int64 device=cpu
  attention_mask   shape=(64, 80) dtype=torch.int64 device=cpu
  tabular          shape=(64, 62) dtype=torch.float32 device=cpu
  label            shape=(64,) dtype=torch.float32 device=cpu
after .to(device): {'input_ids': 'cpu', 'attention_mask': 'cpu', 'tabular': 'cpu', 'label': 'cpu'}


## 6. Model

Forward (hybrid, Arms A/B):

1. `tok_emb = Embedding(vocab_size, d_model)(input_ids) * sqrt(d_model)`
2. Add sinusoidal positional encoding (Vaswani; D5)
3. `nn.TransformerEncoder` (`batch_first=True`, 2 layers, 4 heads, `d_model=64`, FFN 128, dropout 0.1)
4. HF `attention_mask` → `src_key_padding_mask` (`True` where pad)
5. **Masked mean pool over all non-pad positions**, including `[CLS]`, `[SEP]`, and both field separators (D22). Separators mark field boundaries (D14); excluding them would need a second mask.
6. `concat(text_vec, tabular)` → MLP `(d_model + n_tabular)` → 128 → 64 → 1 logit

Arm C (`use_text_encoder=False`): skip embed / PE / encoder / pool; MLP input is `n_tabular` (read from the fitted encoder). Same MLP depths, loss, optimizer, schedule, early stopping.

No sigmoid in the model — `BCEWithLogitsLoss` applies it internally.

| Arm | MLP input | Hidden | Output |
| --- | ---: | --- | ---: |
| A, B (hybrid) | `d_model + n_tabular` | 128 → 64 | 1 |
| C (tabular-only) | `n_tabular` | 128 → 64 | 1 |


In [20]:
class SinusoidalPositionalEncoding(nn.Module):
    """Vaswani et al. sinusoidal PE. Buffer is not a learned parameter."""

    def __init__(self, d_model: int, max_length: int = MAX_LENGTH) -> None:
        super().__init__()
        pe = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10_000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


In [21]:
class BTRHybridModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        n_tabular: int,
        *,
        d_model: int = 64,
        nhead: int = 4,
        num_encoder_layers: int = 2,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
        mlp_dropout: float = 0.2,
        max_length: int = MAX_LENGTH,
        use_text_encoder: bool = True,
        pad_token_id: int = 0,
    ) -> None:
        super().__init__()
        self.use_text_encoder = use_text_encoder
        self.d_model = d_model
        self.n_tabular = n_tabular
        if use_text_encoder:
            self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
            self.pos_enc = SinusoidalPositionalEncoding(d_model, max_length)
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(
                enc_layer,
                num_layers=num_encoder_layers,
                enable_nested_tensor=False,
            )
            mlp_in = d_model + n_tabular
        else:
            mlp_in = n_tabular
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, 128),
            nn.ReLU(),
            nn.Dropout(mlp_dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(mlp_dropout),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        tabular: torch.Tensor,
    ) -> torch.Tensor:
        if self.use_text_encoder:
            x = self.tok_emb(input_ids) * math.sqrt(self.d_model)
            x = self.pos_enc(x)
            key_padding_mask = attention_mask == 0
            h = self.encoder(x, src_key_padding_mask=key_padding_mask)
            # D22: pool ALL non-pad positions, including [CLS], [SEP], field separators.
            mask = attention_mask.unsqueeze(-1).to(dtype=h.dtype)
            denom = mask.sum(dim=1).clamp(min=1.0)
            text_vec = (h * mask).sum(dim=1) / denom
            feat = torch.cat([text_vec, tabular], dim=-1)
        else:
            feat = tabular
        return self.mlp(feat).squeeze(-1)


In [22]:
def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


hybrid_model = BTRHybridModel(
    vocab_size=tokenizer.vocab_size,
    n_tabular=n_tabular,
    use_text_encoder=True,
    pad_token_id=int(tokenizer.pad_token_id),
)
tabular_model = BTRHybridModel(
    vocab_size=tokenizer.vocab_size,
    n_tabular=n_tabular,
    use_text_encoder=False,
    pad_token_id=int(tokenizer.pad_token_id),
)

emb_params = n_params(hybrid_model.tok_emb)
print(f"hybrid params:      {n_params(hybrid_model):,}")
print(f"  embedding table:  {emb_params:,}  ({emb_params / n_params(hybrid_model):.1%} of hybrid)")
print(f"tabular-only params:{n_params(tabular_model):,}")
print(f"MLP in_features hybrid={64 + n_tabular}  tabular={n_tabular}")
assert hybrid_model.mlp[0].in_features == 64 + n_tabular
assert tabular_model.mlp[0].in_features == n_tabular
assert not hasattr(tabular_model, "tok_emb")


hybrid params:      2,044,929
  embedding table:  1,953,408  (95.5% of hybrid)
tabular-only params:16,385
MLP in_features hybrid=126  tabular=62


## 7. Train / eval helpers

- Loss: `BCEWithLogitsLoss` (no `pos_weight` on the first run — D9).
- Optimizer: AdamW, `lr=3e-4`, `weight_decay=1e-2`. Linear warmup over the first 10% of planned steps, then constant. `clip_grad_norm_(1.0)`.
- Why not `lr=1e-3`: `TransformerEncoderLayer` is post-LN by default; that configuration is known to need warmup (D18). Arm C keeps the same settings so the arms stay comparable.
- Early stopping monitors **validation PR-AUC** (mode max, patience 5, restore best weights).
- Each epoch logs train loss and **train and valid** ROC-AUC and PR-AUC. Train curves are required by the §7 diagnostic, not optional.
- `run_arm` repeats 5 seeds on the **same** split and returns mean ± sd (D16). It is defined here and signature-checked; it is **not** executed in this notebook revision.


In [23]:
def logits_to_proba(logits: torch.Tensor) -> torch.Tensor:
    return torch.sigmoid(logits)


def batch_to_device(
    batch: dict[str, torch.Tensor],
    device: torch.device,
) -> dict[str, torch.Tensor]:
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def compute_aucs(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, float]:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if np.unique(y_true).size < 2:
        return float("nan"), float("nan")
    return (
        float(roc_auc_score(y_true, y_score)),
        float(average_precision_score(y_true, y_score)),
    )


def per_query_metrics(
    y_true: np.ndarray,
    y_score: np.ndarray,
    query_ids: np.ndarray,
) -> tuple[float, float]:
    """Mean AP and recall@1 over queries with at least one purchase (D23)."""
    frame = pd.DataFrame({"y": y_true, "s": y_score, "q": query_ids})
    aps: list[float] = []
    r1s: list[float] = []
    for _, g in frame.groupby("q", sort=False):
        if g["y"].sum() < 1:
            continue
        aps.append(float(average_precision_score(g["y"].to_numpy(), g["s"].to_numpy())))
        r1s.append(float(g.loc[g["s"].idxmax(), "y"]))
    if not aps:
        return float("nan"), float("nan")
    return float(np.mean(aps)), float(np.mean(r1s))


# metric helpers smoke
_yt = np.array([0.0, 1.0, 0.0, 1.0])
_ys = np.array([0.1, 0.9, 0.2, 0.8])
_roc, _pr = compute_aucs(_yt, _ys)
assert _roc > 0.9 and _pr > 0.9
_ap, _r1 = per_query_metrics(_yt, _ys, np.array(["a", "a", "b", "b"]))
assert _r1 == 1.0
print(f"compute_aucs smoke  ROC={_roc:.3f} PR={_pr:.3f}")
print(f"per_query_metrics smoke  AP={_ap:.3f} recall@1={_r1:.3f}")


compute_aucs smoke  ROC=1.000 PR=1.000
per_query_metrics smoke  AP=1.000 recall@1=1.000


In [24]:
def make_warmup_scheduler(
    optimizer: torch.optim.Optimizer,
    total_steps: int,
) -> torch.optim.lr_scheduler.LambdaLR:
    warmup_steps = max(1, int(0.10 * total_steps))

    def lr_lambda(current_step: int) -> float:
        if current_step < warmup_steps:
            return float(current_step + 1) / float(warmup_steps)
        return 1.0

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    loss_fn: nn.Module,
    device: torch.device,
) -> tuple[float, float, float]:
    model.train()
    total_loss = 0.0
    n = 0
    ys: list[np.ndarray] = []
    scores: list[np.ndarray] = []
    for batch in loader:
        batch = batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch["input_ids"], batch["attention_mask"], batch["tabular"])
        loss = loss_fn(logits, batch["label"])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        bs = int(batch["label"].shape[0])
        total_loss += float(loss.item()) * bs
        n += bs
        ys.append(batch["label"].detach().cpu().numpy())
        scores.append(logits_to_proba(logits).detach().cpu().numpy())
    roc, pr = compute_aucs(np.concatenate(ys), np.concatenate(scores))
    return total_loss / n, roc, pr


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
    device: torch.device,
) -> tuple[float, float, float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    n = 0
    ys: list[np.ndarray] = []
    scores: list[np.ndarray] = []
    for batch in loader:
        batch = batch_to_device(batch, device)
        logits = model(batch["input_ids"], batch["attention_mask"], batch["tabular"])
        loss = loss_fn(logits, batch["label"])
        bs = int(batch["label"].shape[0])
        total_loss += float(loss.item()) * bs
        n += bs
        ys.append(batch["label"].detach().cpu().numpy())
        scores.append(logits_to_proba(logits).detach().cpu().numpy())
    y_true = np.concatenate(ys)
    y_score = np.concatenate(scores)
    roc, pr = compute_aucs(y_true, y_score)
    return total_loss / n, roc, pr, y_true, y_score


In [25]:
def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fit(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    device: torch.device,
    *,
    max_epochs: int = MAX_EPOCHS,
    patience: int = PATIENCE,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    checkpoint_path: str | None = None,
) -> tuple[dict[str, list[float]], int]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = max_epochs * max(1, len(train_loader))
    scheduler = make_warmup_scheduler(optimizer, total_steps)
    loss_fn = nn.BCEWithLogitsLoss()
    history: dict[str, list[float]] = {
        "train_loss": [],
        "valid_loss": [],
        "train_roc": [],
        "valid_roc": [],
        "train_pr": [],
        "valid_pr": [],
    }
    best_pr = -1.0
    best_state: dict[str, torch.Tensor] | None = None
    best_epoch = 0
    wait = 0
    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_roc, tr_pr = train_one_epoch(
            model, train_loader, optimizer, scheduler, loss_fn, device
        )
        va_loss, va_roc, va_pr, _, _ = evaluate(model, valid_loader, loss_fn, device)
        history["train_loss"].append(tr_loss)
        history["valid_loss"].append(va_loss)
        history["train_roc"].append(tr_roc)
        history["valid_roc"].append(va_roc)
        history["train_pr"].append(tr_pr)
        history["valid_pr"].append(va_pr)
        print(
            f"epoch {epoch:02d}  train loss={tr_loss:.4f} roc={tr_roc:.3f} pr={tr_pr:.3f}  "
            f"valid loss={va_loss:.4f} roc={va_roc:.3f} pr={va_pr:.3f}"
        )
        if va_pr > best_pr:
            best_pr = va_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            wait = 0
            if checkpoint_path is not None:
                torch.save(best_state, checkpoint_path)
        else:
            wait += 1
            if wait >= patience:
                print(f"early stop at epoch {epoch}; best epoch={best_epoch} valid PR-AUC={best_pr:.3f}")
                break
    if best_state is None:
        raise RuntimeError("fit() produced no checkpoint")
    model.load_state_dict(best_state)
    return history, best_epoch


In [26]:
def run_arm(
    *,
    use_text_encoder: bool,
    text_variant: str,
    seeds: Sequence[int] = ARM_SEEDS,
    max_epochs: int = MAX_EPOCHS,
    checkpoint_path: str | None = None,
) -> dict[str, float]:
    """Train one arm over `seeds` on the locked split; return mean ± sd. Not executed here."""
    rows: list[dict[str, float]] = []
    for seed in seeds:
        seed_all(int(seed))
        model = BTRHybridModel(
            vocab_size=tokenizer.vocab_size,
            n_tabular=n_tabular,
            use_text_encoder=use_text_encoder,
            pad_token_id=int(tokenizer.pad_token_id),
        ).to(device)
        history, best_epoch = fit(
            model,
            make_loader("train", text_variant),
            make_loader("valid", text_variant, shuffle=False),
            device,
            max_epochs=max_epochs,
            checkpoint_path=checkpoint_path,
        )
        rows.append(
            {
                "seed": float(seed),
                "best_epoch": float(best_epoch),
                "valid_roc": history["valid_roc"][best_epoch - 1],
                "valid_pr": history["valid_pr"][best_epoch - 1],
            }
        )
    table = pd.DataFrame(rows)
    return {
        "valid_roc_mean": float(table["valid_roc"].mean()),
        "valid_roc_sd": float(table["valid_roc"].std(ddof=1)) if len(table) > 1 else 0.0,
        "valid_pr_mean": float(table["valid_pr"].mean()),
        "valid_pr_sd": float(table["valid_pr"].std(ddof=1)) if len(table) > 1 else 0.0,
        "best_epoch_mean": float(table["best_epoch"].mean()),
        "n_seeds": float(len(table)),
    }


sig = inspect.signature(run_arm)
for required in ("use_text_encoder", "text_variant", "seeds"):
    assert required in sig.parameters, required
print("run_arm signature OK:", sig)
print("run_arm is defined; not executed (no training run in this revision).")


run_arm signature OK: (*, use_text_encoder: 'bool', text_variant: 'str', seeds: 'Sequence[int]' = (42, 43, 44, 45, 46), max_epochs: 'int' = 30, checkpoint_path: 'str | None' = None) -> 'dict[str, float]'
run_arm is defined; not executed (no training run in this revision).


In [27]:
loss_fn = nn.BCEWithLogitsLoss()
batch = batch_to_device(next(iter(make_loader("train", "full", shuffle=False))), device)

print("=== one-batch forward/backward smoke ===")
for name, model in ("hybrid", hybrid_model), ("tabular-only", tabular_model):
    model = model.to(device)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    opt.zero_grad(set_to_none=True)
    logits = model(batch["input_ids"], batch["attention_mask"], batch["tabular"])
    assert logits.shape == batch["label"].shape, (logits.shape, batch["label"].shape)
    loss = loss_fn(logits, batch["label"])
    assert torch.isfinite(loss), loss
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
    opt.step()
    print(
        f"{name:14s}  logits={tuple(logits.shape)}  loss={float(loss.item()):.4f}  "
        f"grad_norm={float(grad_norm):.4f}  params={n_params(model):,}"
    )

print("smoke test passed — ready to train.")


=== one-batch forward/backward smoke ===
hybrid          logits=(64,)  loss=0.7052  grad_norm=0.5457  params=2,044,929
tabular-only    logits=(64,)  loss=0.6724  grad_norm=0.5016  params=16,385
smoke test passed — ready to train.


## Placeholder — cell groups 8–12 (not in this revision)

The training machinery above is complete. The following stay for a later notebook pass and must **not** be run against the test set until they exist:

- **8 — Reference baselines:** prevalence predictor, logistic regression and gradient boosting on the tabular block, title-cue one-hot logistic regression. Same grouped split.
- **9 — Three arms:** A (full text), B (cue-stripped), C (tabular-only), each over 5 seeds. Cue-stripping TF-IDF verification probe. Train/valid curves. Test once per arm on the best checkpoint.
- **10 — Discussion:** A vs C is cue visibility; B vs C is the language claim. Limitations from `DESIGN.md` §10.
- **11 — Architecture scale-up (required):** at least two larger configs on Arms A and B, selected on **valid** only. Test stays empty for exploratory configs.
- **12 — Exercise 3:** personalization design sketch (this CSV has no user identifier).
